In [ ]:
!pip install -q --upgrade pip
!pip install -q numpy==2.0.2 scipy==1.13.0 pandas==2.2.2 numba==0.60.0
!pip install -q torch torchaudio torchvision
!pip install -q librosa==0.10.2 soundfile
!pip install -q scikit-learn matplotlib seaborn tqdm
!pip install -q datasets huggingface_hub

print("\n All dependencies installed")

## Google Drive & HuggingFace Setup

- Drive stores checkpoints and forensic reports.
- IndicVoices is a **gated** dataset — you need a HuggingFace token with accepted terms.
  Get one at: https://huggingface.co/settings/tokens
  Then accept dataset terms at: https://huggingface.co/datasets/ai4bharat/IndicVoices

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/VoiceTrustPlatform'
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/forensics',   exist_ok=True)
print(f" Drive mounted. Outputs → {DRIVE_PATH}")

In [ ]:
# ── HuggingFace login (required for gated IndicVoices dataset) ────────────────
from huggingface_hub import login
from google.colab import userdata

# Option A: auto-read from Colab Secrets (Colab → Tools → Secrets → add HF_TOKEN)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(" Logged in via Colab secret")
except Exception:
    # Option B: paste manually
    HF_TOKEN = input("Paste your HuggingFace token (hf_...): ").strip()
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(" Logged in manually")

## Imports & Device

In [ ]:
import os, json, glob, warnings, time, random, shutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
warnings.filterwarnings('ignore')

import numpy as np
import librosa
import scipy.signal as signal
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchaudio
import torchaudio.transforms as T

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm

from datasets import load_dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## Config

In [ ]:
@dataclass
class Config:
    # Paths
    DRIVE_PATH:    str   = '/content/drive/MyDrive/VoiceTrustPlatform'
    AUDIO_DIR:     str   = '/content/indicvoices_audio'
    CACHE_DIR:     str   = '/content/feature_cache'

    # Audio
    SAMPLE_RATE:   int   = 16000
    MAX_DURATION:  float = 4.0
    MAX_SAMPLES:   int   = 64000   # 4s × 16kHz

    # Mel
    N_MELS:        int   = 64
    N_FFT:         int   = 400
    HOP_LENGTH:    int   = 160
    F_MIN:         int   = 0
    F_MAX:         int   = 8000

    # CQCC
    CQT_BINS:      int   = 84
    CQCC_COEFFS:   int   = 20

    # Model
    CNN_CHANNELS:  tuple = (32, 64, 128, 256)
    EMB_DIM:       int   = 128
    AASIST_HEADS:  int   = 4

    # Training
    BATCH_SIZE:    int   = 32
    NUM_WORKERS:   int   = 2
    EPOCHS:        int   = 30
    LR:            float = 1e-3
    WEIGHT_DECAY:  float = 1e-4
    PATIENCE:      int   = 7

    # Risk
    RISK_LOW:      float = 0.35
    RISK_HIGH:     float = 0.65

    DEVICE:        str   = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = Config()
os.makedirs(cfg.AUDIO_DIR, exist_ok=True)
os.makedirs(cfg.CACHE_DIR, exist_ok=True)
print(" Config ready")

## IndicVoices Dataset — Real Speech (Label = 0 / REAL)

[`ai4bharat/IndicVoices`](https://huggingface.co/datasets/ai4bharat/IndicVoices) contains
natural, spontaneous speech from 51K speakers across 22 Indian languages. All samples here
are **genuine human speech** → label = 0 (REAL).

Supported language subsets (config names used by the `datasets` library):
`assamese`, `bengali`, `bodo`, `dogri`, `gujarati`, `hindi`, `kannada`, `kashmiri`,
`konkani`, `maithili`, `malayalam`, `manipuri`, `marathi`, `nepali`, `odia`,
`punjabi`, `sanskrit`, `santali`, `sindhi`, `tamil`, `telugu`, `urdu`


In [ ]:
# Languages to load (all 22; remove any if short on Colab disk)
INDIC_LANGS = [
    "assamese", "bengali", "bodo", "dogri", "gujarati", "hindi",
    "kannada", "kashmiri", "konkani", "maithili", "malayalam", "manipuri",
    "marathi", "nepali", "odia", "punjabi", "sanskrit", "santali",
    "sindhi", "tamil", "telugu", "urdu",
]

MAX_PER_LANG   = 300    # 22 × 300 = 6 600 real samples  (adjust to Colab RAM/disk)
SPLIT_TO_USE   = "train"   # "train" | "valid" | "test"

real_records   = []

for lang in INDIC_LANGS:
    print(f"\n  Loading IndicVoices → {lang} ...")
    try:
        ds = load_dataset(
            "ai4bharat/IndicVoices",
            lang,
            split=SPLIT_TO_USE,
            streaming=True,
            trust_remote_code=True,
        )
    except Exception as e:
        print(f"  [SKIP] {lang}: {e}")
        continue

    lang_dir = os.path.join(cfg.AUDIO_DIR, "real", lang)
    os.makedirs(lang_dir, exist_ok=True)
    count = 0

    for i, ex in enumerate(ds):
        if count >= MAX_PER_LANG:
            break

        audio = ex.get("audio")
        if audio is None:
            continue

        file_id  = f"iv_{lang}_{i}"
        out_path = os.path.join(lang_dir, f"{file_id}.wav")

        try:
            arr = np.array(audio["array"], dtype=np.float32)
            sr  = int(audio["sampling_rate"])
            # Resample to 16 kHz if needed
            if sr != cfg.SAMPLE_RATE:
                arr = librosa.resample(arr, orig_sr=sr, target_sr=cfg.SAMPLE_RATE)
            sf.write(out_path, arr, cfg.SAMPLE_RATE)
        except Exception as e:
            continue

        real_records.append({
            "speaker_id":  str(ex.get("speaker_id", f"{lang}_{i}")),
            "file_id":     file_id,
            "language":    lang,
            "attack_type": "-",
            "label":       0,          # REAL genuine human speech
            "label_str":   "bonafide",
            "audio_path":  out_path,
        })
        count += 1

    print(f"   {lang}: {count} real samples saved")

print(f"\n Total REAL records: {len(real_records)}")

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "ai4bharat/IndicVoices",
    "hindi",
    split="train[:100]"
)

print(ds)

## Fake / Spoof Class Generation

IndicVoices contains only genuine speech, so we synthesise a **spoof class** by applying
voice-conversion-style augmentations to the real audio:

| Augmentation | Effect |
|---|---|
| Pitch shift (+4 semitones) | simulates TTS / voice-cloning pitch offset |
| Speed perturbation (0.85×) | simulates replay / codec artefact |
| Both combined | harder synthetic sample |

Each real file generates **one fake counterpart**, keeping the dataset balanced 1:1.


In [ ]:
# Generate balanced fake/spoof class from real IndicVoices audio

fake_records = []
fake_dir     = os.path.join(cfg.AUDIO_DIR, "fake")
os.makedirs(fake_dir, exist_ok=True)

AUG_VARIANTS = [
    ("pitch",  dict(n_steps=4)),          # pitch-shift +4 semitones
    ("speed",  dict(rate=0.85)),          # slow-down replay artefact
    ("combo",  dict(n_steps=3, rate=0.9)),# combined
]

print(f"Generating {len(real_records)} fake samples (1:1 balance) ...")

for rec in tqdm(real_records, desc="Creating fake samples"):
    try:
        wav, sr = librosa.load(rec["audio_path"], sr=cfg.SAMPLE_RATE)

        # Pick augmentation variant deterministically from file_id hash
        variant_name, params = AUG_VARIANTS[hash(rec["file_id"]) % len(AUG_VARIANTS)]

        if variant_name == "pitch":
            wav_fake = librosa.effects.pitch_shift(wav, sr=sr, n_steps=params["n_steps"])
            attack   = "TTS_PITCH_SHIFT"
        elif variant_name == "speed":
            wav_fake = librosa.effects.time_stretch(wav, rate=params["rate"])
            attack   = "REPLAY_SPEED_PERTURB"
        else:  # combo
            wav_fake = librosa.effects.pitch_shift(wav, sr=sr, n_steps=params["n_steps"])
            wav_fake = librosa.effects.time_stretch(wav_fake, rate=params["rate"])
            attack   = "VOICE_CONVERSION_COMBO"

        # Trim/pad to same length as original
        if len(wav_fake) > len(wav):
            wav_fake = wav_fake[:len(wav)]
        else:
            wav_fake = np.pad(wav_fake, (0, max(0, len(wav) - len(wav_fake))))

        file_id  = f"fake_{rec['file_id']}"
        out_path = os.path.join(fake_dir, f"{file_id}.wav")
        sf.write(out_path, wav_fake.astype(np.float32), cfg.SAMPLE_RATE)

        fake_records.append({
            "speaker_id":  rec["speaker_id"],
            "file_id":     file_id,
            "language":    rec["language"],
            "attack_type": attack,
            "label":       1,          # FAKE — augmented/synthetic
            "label_str":   "spoof",
            "audio_path":  out_path,
        })
    except Exception:
        continue

print(f"\n Total FAKE records: {len(fake_records)}")
print(f" Real : Fake = {len(real_records)} : {len(fake_records)}")

## Build Train / Dev / Eval Splits (70 / 15 / 15)

In [ ]:
# Merge + shuffle + stratified split
random.seed(42)

combined = real_records + fake_records
random.shuffle(combined)

n       = len(combined)
n_train = int(0.70 * n)
n_dev   = int(0.15 * n)

train_records = combined[:n_train]
dev_records   = combined[n_train : n_train + n_dev]
eval_records  = combined[n_train + n_dev :]

def class_counts(recs):
    r = sum(r['label']==0 for r in recs)
    f = sum(r['label']==1 for r in recs)
    return f"real={r} fake={f}"

print(f"\n Dataset splits:")
print(f"  Train : {len(train_records):>5}  ({class_counts(train_records)})")
print(f"  Dev   : {len(dev_records):>5}  ({class_counts(dev_records)})")
print(f"  Eval  : {len(eval_records):>5}  ({class_counts(eval_records)})")

In [ ]:
import numpy as np, numba, librosa, scipy, torch
print("NumPy :", np.__version__)
print("Numba :", numba.__version__)
print("librosa:", librosa.__version__)
print("scipy :", scipy.__version__)
print("torch :", torch.__version__)
print(" Everything working")

## Audio Preprocessor

In [ ]:
class AudioPreprocessor:
    """Resampling, normalization, pad/trim, Log-Mel + CQCC features."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.mel_transform = T.MelSpectrogram(
            sample_rate=cfg.SAMPLE_RATE, n_fft=cfg.N_FFT,
            hop_length=cfg.HOP_LENGTH,   n_mels=cfg.N_MELS,
            f_min=cfg.F_MIN,             f_max=cfg.F_MAX,
        )
        self.amp_to_db = T.AmplitudeToDB(stype='power')

    def load_audio(self, path: str) -> Optional[torch.Tensor]:
        try:
            wav, sr = torchaudio.load(path)
            if wav.shape[0] > 1:
                wav = wav.mean(dim=0, keepdim=True)
            if sr != self.cfg.SAMPLE_RATE:
                wav = torchaudio.functional.resample(wav, sr, self.cfg.SAMPLE_RATE)
            return wav
        except Exception:
            return None

    def pad_or_trim(self, wav: torch.Tensor) -> torch.Tensor:
        T_len = wav.shape[-1]
        max_s = self.cfg.MAX_SAMPLES
        if T_len < max_s:
            repeats = (max_s // T_len) + 1
            wav = wav.repeat(1, repeats)[:, :max_s]
        else:
            wav = wav[:, :max_s]
        return wav

    def normalize(self, wav: torch.Tensor) -> torch.Tensor:
        peak = wav.abs().max()
        return wav / (peak + 1e-8)

    def log_mel(self, wav: torch.Tensor) -> torch.Tensor:
        mel     = self.mel_transform(wav)
        log_mel = self.amp_to_db(mel)
        return (log_mel - log_mel.mean()) / (log_mel.std() + 1e-8)

    def cqcc(self, wav: torch.Tensor) -> np.ndarray:
        y = wav.squeeze().numpy()
        cqt = np.abs(librosa.cqt(
            y, sr=self.cfg.SAMPLE_RATE, hop_length=self.cfg.HOP_LENGTH,
            n_bins=self.cfg.CQT_BINS, bins_per_octave=24,
        ))
        log_cqt   = librosa.power_to_db(cqt ** 2)
        cqcc_feat = librosa.feature.mfcc(S=log_cqt, n_mfcc=self.cfg.CQCC_COEFFS)
        target_len = self.cfg.MAX_SAMPLES // self.cfg.HOP_LENGTH + 1
        if cqcc_feat.shape[1] < target_len:
            cqcc_feat = np.pad(cqcc_feat, ((0,0),(0, target_len - cqcc_feat.shape[1])))
        else:
            cqcc_feat = cqcc_feat[:, :target_len]
        return cqcc_feat.astype(np.float32)

    def process(self, path: str) -> Optional[Dict]:
        wav = self.load_audio(path)
        if wav is None:
            return None
        wav = self.normalize(self.pad_or_trim(wav))
        return {
            'wav':     wav,
            'log_mel': self.log_mel(wav),
            'cqcc':    torch.from_numpy(self.cqcc(wav)).unsqueeze(0),
        }


preprocessor = AudioPreprocessor(cfg)
print(' Preprocessor ready')

if train_records:
    feat = preprocessor.process(train_records[0]['audio_path'])
    if feat:
        print(f'  Mel shape : {feat["log_mel"].shape}')
        print(f'  CQCC shape: {feat["cqcc"].shape}')

## PyTorch Dataset & DataLoader

In [ ]:
class IndicVoicesDataset(Dataset):
    def __init__(self, records: List[Dict], preprocessor: AudioPreprocessor,
                 cache_dir: str = None):
        self.records   = records
        self.prep      = preprocessor
        self.cache_dir = Path(cache_dir) if cache_dir else None
        if self.cache_dir:
            self.cache_dir.mkdir(parents=True, exist_ok=True)

    def __len__(self):
        return len(self.records)

    def _cache_path(self, file_id: str) -> Path:
        return self.cache_dir / f'{file_id}.pt'

    def __getitem__(self, idx):
        rec    = self.records[idx]
        label  = rec['label']
        attack = rec.get('attack_type', '-')

        if self.cache_dir:
            cp = self._cache_path(rec['file_id'])
            if cp.exists():
                try:
                    cached = torch.load(cp, weights_only=True)
                    return cached['mel'], cached['cqcc'], label, attack
                except Exception:
                    pass

        feats = self.prep.process(rec['audio_path'])
        if feats is None:
            mel_shape  = (1, cfg.N_MELS,      cfg.MAX_SAMPLES // cfg.HOP_LENGTH + 1)
            cqcc_shape = (1, cfg.CQCC_COEFFS, cfg.MAX_SAMPLES // cfg.HOP_LENGTH + 1)
            return torch.zeros(mel_shape), torch.zeros(cqcc_shape), label, attack

        mel  = feats['log_mel']
        cqcc = feats['cqcc']
        if self.cache_dir:
            torch.save({'mel': mel, 'cqcc': cqcc}, self._cache_path(rec['file_id']))
        return mel, cqcc, label, attack


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    mels, cqccs, labels, attacks = zip(*batch)
    return (
        torch.stack(mels),
        torch.stack(cqccs),
        torch.tensor(labels, dtype=torch.long),
        list(attacks),
    )

def make_loader(records, shuffle=True) -> DataLoader:
    ds = IndicVoicesDataset(records, preprocessor, cache_dir=cfg.CACHE_DIR)
    return DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, collate_fn=collate_fn,
                      pin_memory=torch.cuda.is_available())

train_loader = make_loader(train_records, shuffle=True)
dev_loader   = make_loader(dev_records,   shuffle=False)
eval_loader  = make_loader(eval_records,  shuffle=False)
print(f' DataLoaders ready | Train={len(train_loader)} Dev={len(dev_loader)} Eval={len(eval_loader)} batches')

## CNN Baseline Model (SE + Residual Blocks)

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation attention."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.se(x).view(-1, x.shape[1], 1, 1)


class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, use_se: bool = True):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU(),
        )
        self.se   = SEBlock(out_ch) if use_se else nn.Identity()
        self.pool = nn.MaxPool2d(2, 2)
        self.skip = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        out = self.se(self.conv(x)) + F.avg_pool2d(self.skip(x), 1)
        return self.pool(out)


class CNNBaseline(nn.Module):
    """CNN + SE attention baseline. Input: [B,1,N_MELS,T] → logits [B,2] + emb [B,EMB_DIM]"""
    def __init__(self, cfg: Config):
        super().__init__()
        channels = cfg.CNN_CHANNELS
        self.stem = nn.Sequential(
            nn.Conv2d(1, channels[0], 3, padding=1, bias=False),
            nn.BatchNorm2d(channels[0]), nn.GELU(),
        )
        self.blocks = nn.ModuleList([
            ConvBlock(channels[i], channels[i+1]) for i in range(len(channels)-1)
        ])
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.emb = nn.Sequential(
            nn.Flatten(), nn.Linear(channels[-1], cfg.EMB_DIM),
            nn.LayerNorm(cfg.EMB_DIM), nn.GELU(), nn.Dropout(0.3),
        )
        self.cls = nn.Linear(cfg.EMB_DIM, 2)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.stem(x)
        for blk in self.blocks:
            x = blk(x)
        emb = self.emb(self.gap(x))
        return self.cls(emb), emb


cnn_model = CNNBaseline(cfg).to(DEVICE)
print(f' CNN Baseline | Params: {sum(p.numel() for p in cnn_model.parameters())/1e6:.2f}M')

## AASIST-Compatible Architecture

In [ ]:
class GraphAttentionLayer(nn.Module):
    """Simplified graph attention (AASIST-style spectro-temporal)."""
    def __init__(self, in_dim: int, out_dim: int, heads: int = 4):
        super().__init__()
        self.heads  = heads
        self.d_head = out_dim // heads
        self.q = nn.Linear(in_dim, out_dim)
        self.k = nn.Linear(in_dim, out_dim)
        self.v = nn.Linear(in_dim, out_dim)
        self.o = nn.Linear(out_dim, out_dim)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape
        Q = self.q(x).view(B, T, self.heads, self.d_head).transpose(1, 2)
        K = self.k(x).view(B, T, self.heads, self.d_head).transpose(1, 2)
        V = self.v(x).view(B, T, self.heads, self.d_head).transpose(1, 2)
        attn = torch.softmax(Q @ K.transpose(-2,-1) / self.d_head**0.5, dim=-1)
        out  = (attn @ V).transpose(1,2).contiguous().view(B, T, -1)
        return self.norm(x + self.o(out)), attn


class AASISTEncoder(nn.Module):
    """AASIST-compatible spectro-temporal graph encoder."""
    def __init__(self, in_channels=1, base_dim=64, n_heads=4, emb_dim=128):
        super().__init__()
        self.spec_enc = nn.Sequential(
            nn.Conv2d(in_channels, base_dim,   (7,1), padding=(3,0), bias=False),
            nn.BatchNorm2d(base_dim), nn.GELU(),
            nn.Conv2d(base_dim, base_dim*2, (1,7), padding=(0,3), bias=False),
            nn.BatchNorm2d(base_dim*2), nn.GELU(),
            nn.AdaptiveAvgPool2d((8, None)),
        )
        self.proj  = nn.Linear(base_dim * 2 * 8, emb_dim)
        self.gat1  = GraphAttentionLayer(emb_dim, emb_dim, heads=n_heads)
        self.gat2  = GraphAttentionLayer(emb_dim, emb_dim, heads=n_heads)
        self.pool  = nn.AdaptiveAvgPool1d(1)
        self.head  = nn.Sequential(
            nn.Linear(emb_dim, emb_dim//2), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(emb_dim//2, 2),
        )
        self.attn_weights = None

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B = x.shape[0]
        feat = self.spec_enc(x)
        feat = feat.permute(0,3,1,2).contiguous().view(B, feat.shape[3], -1)
        feat = self.proj(feat)
        feat, _    = self.gat1(feat)
        feat, attn = self.gat2(feat)
        self.attn_weights = attn.detach()
        emb = self.pool(feat.transpose(1,2)).squeeze(-1)
        return self.head(emb), emb


aasist_model = AASISTEncoder(
    in_channels=1, base_dim=64,
    n_heads=cfg.AASIST_HEADS, emb_dim=cfg.EMB_DIM,
).to(DEVICE)
print(f' AASIST-compatible | Params: {sum(p.numel() for p in aasist_model.parameters())/1e6:.2f}M')

## Training Pipeline

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-4):
        self.patience  = patience
        self.min_delta = min_delta
        self.counter   = 0
        self.best      = None
        self.stop      = False

    def __call__(self, val_loss):
        if self.best is None or val_loss < self.best - self.min_delta:
            self.best    = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


def train_epoch(model, loader, optimizer, criterion, scaler) -> Dict:
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc='Train', leave=False)
    for batch in pbar:
        if batch is None: continue
        mel, cqcc, labels, _ = batch
        mel, labels = mel.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            logits, _ = model(mel)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    return {'loss': total_loss / max(len(loader),1), 'acc': correct / max(total,1)}


@torch.no_grad()
def eval_epoch(model, loader, criterion) -> Dict:
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    for batch in tqdm(loader, desc='Eval', leave=False):
        if batch is None: continue
        mel, cqcc, labels, _ = batch
        mel, labels = mel.to(DEVICE), labels.to(DEVICE)
        with autocast():
            logits, _ = model(mel)
            loss = criterion(logits, labels)
        total_loss += loss.item()
        probs = F.softmax(logits, dim=1)[:, 1]
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())
    return {'loss': total_loss / max(len(loader),1),
            'preds': all_preds, 'labels': all_labels, 'probs': all_probs}


def compute_eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores)
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    return (fpr[idx] + fnr[idx]) / 2


def full_metrics(result) -> Dict:
    y, p, s = result['labels'], result['preds'], result['probs']
    return {
        'accuracy':  round(accuracy_score(y, p), 4),
        'precision': round(precision_score(y, p, zero_division=0), 4),
        'recall':    round(recall_score(y, p, zero_division=0), 4),
        'f1':        round(f1_score(y, p, zero_division=0), 4),
        'roc_auc':   round(roc_auc_score(y, s) if len(set(y)) > 1 else 0.5, 4),
        'eer':       round(compute_eer(y, s)    if len(set(y)) > 1 else 0.5, 4),
    }


def train(model, train_loader, val_loader, name='model') -> Dict:
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS, eta_min=1e-6)
    scaler    = GradScaler()
    stopper   = EarlyStopping(patience=cfg.PATIENCE)
    history   = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    best_val  = float('inf')
    ckpt_path = f'{cfg.DRIVE_PATH}/checkpoints/{name}_best.pt'

    for epoch in range(1, cfg.EPOCHS + 1):
        t = train_epoch(model, train_loader, optimizer, criterion, scaler)
        v = eval_epoch(model, val_loader, criterion)
        scheduler.step()
        m = full_metrics(v)
        history['train_loss'].append(t['loss'])
        history['val_loss'].append(v['loss'])
        history['train_acc'].append(t['acc'])
        history['val_acc'].append(m['accuracy'])
        print(f"Epoch {epoch:3d}/{cfg.EPOCHS} | "
              f"TLoss={t['loss']:.4f} TAcc={t['acc']:.4f} | "
              f"VLoss={v['loss']:.4f} VAcc={m['accuracy']:.4f} | "
              f"F1={m['f1']:.4f} EER={m['eer']:.4f}")
        if v['loss'] < best_val:
            best_val = v['loss']
            torch.save({'epoch': epoch, 'model': model.state_dict(), 'metrics': m}, ckpt_path)
            print(f"   Saved → {ckpt_path}")
        if stopper(v['loss']):
            print(f"   Early stopping at epoch {epoch}")
            break
    return history

print(' Training pipeline ready')

## Run Training

In [ ]:
if train_loader and dev_loader:
    print('=== Training CNN Baseline ===')
    cnn_history = train(cnn_model, train_loader, dev_loader, name='cnn_baseline')

    print('\n=== Training AASIST ===')
    aasist_history = train(aasist_model, train_loader, dev_loader, name='aasist')
else:
    print('No training data — skipping (demo mode)')

## Evaluation Pipeline

In [ ]:
def evaluate_and_report(model, loader, model_name: str = 'Model'):
    criterion = nn.CrossEntropyLoss()
    result    = eval_epoch(model, loader, criterion)
    metrics   = full_metrics(result)

    print(f'\n{"="*50}')
    print(f'  {model_name} Evaluation Report')
    print(f'{"="*50}')
    for k, v in metrics.items():
        print(f'  {k:12s}: {v}')

    cm = confusion_matrix(result['labels'], result['preds'])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Real','Fake'], yticklabels=['Real','Fake'])
    axes[0].set_title(f'{model_name} — Confusion Matrix')
    axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

    if len(set(result['labels'])) > 1:
        fpr, tpr, _ = roc_curve(result['labels'], result['probs'])
        axes[1].plot(fpr, tpr, label=f"AUC={metrics['roc_auc']:.3f}", color='steelblue', lw=2)
        axes[1].plot([0,1],[0,1], 'k--', lw=1)
        axes[1].set_title(f'{model_name} — ROC Curve')
        axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
        axes[1].legend()

    plt.tight_layout()
    plt.savefig(f'{cfg.DRIVE_PATH}/forensics/{model_name}_eval.png', dpi=150)
    plt.show()
    return metrics

if eval_loader:
    cnn_metrics    = evaluate_and_report(cnn_model,    eval_loader, 'CNN_Baseline')
    aasist_metrics = evaluate_and_report(aasist_model, eval_loader, 'AASIST')

## Explainability Engine GradCAM + Spectral Forensics

In [ ]:
import torch.nn.functional as F


class GradCAM:
    """Gradient-weighted Class Activation Maps for CNN."""

    def __init__(self, model: CNNBaseline, target_layer_idx: int = -1):
        self.model       = model
        self.gradients   = None
        self.activations = None
        target = model.blocks[target_layer_idx].conv
        target.register_forward_hook(self._save_activation)
        target.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, _, __, output):
        self.activations = output.detach()

    def _save_gradient(self, _, __, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, x: torch.Tensor, class_idx: int = 1) -> np.ndarray:
        self.model.eval()
        x = x.to(DEVICE).requires_grad_(True)
        logits, _ = self.model(x)
        self.model.zero_grad()
        logits[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2,3), keepdim=True)
        cam     = (weights * self.activations).sum(dim=1).squeeze()
        cam     = F.relu(cam).cpu().numpy()
        return cam / (cam.max() + 1e-8)


class SpectralAnomalyAnalyzer:
    """Forensic spectral analysis — detect deepfake artefacts."""

    def __init__(self, cfg: Config):
        self.cfg = cfg

    def analyze(self, wav: np.ndarray, sr: int = 16000) -> Dict:
        ind = {}
        flatness = librosa.feature.spectral_flatness(y=wav)
        ind['spectral_flatness_mean'] = float(flatness.mean())
        ind['high_flatness']          = bool(flatness.mean() > 0.15)

        rolloff = librosa.feature.spectral_rolloff(y=wav, sr=sr)
        ind['rolloff_mean'] = float(rolloff.mean())
        ind['low_rolloff']  = bool(rolloff.mean() < 4000)

        zcr = librosa.feature.zero_crossing_rate(wav)
        ind['zcr_std']                = float(zcr.std())
        ind['unnatural_zcr_variance'] = bool(zcr.std() < 0.02)

        f0, _, _ = librosa.pyin(wav, fmin=80, fmax=400, sr=sr)
        f0v = f0[~np.isnan(f0)] if f0 is not None else np.array([])
        ind['pitch_std']     = float(np.std(f0v)) if len(f0v) > 10 else 0.0
        ind['robotic_pitch'] = bool(ind['pitch_std'] < 8.0 and len(f0v) > 10)

        harmonic = librosa.effects.harmonic(wav)
        noise    = wav - harmonic
        hnr = 10 * np.log10((harmonic**2).mean() / ((noise**2).mean() + 1e-8) + 1e-8)
        ind['hnr']     = float(hnr)
        ind['low_hnr'] = bool(hnr < 5.0)

        stft  = librosa.stft(wav, n_fft=512)
        phase = np.angle(stft)
        gd    = float(np.diff(phase, axis=1).var())
        ind['phase_variance']  = gd
        ind['unnatural_phase'] = bool(gd < 0.5 or gd > 10.0)

        intervals  = librosa.effects.split(wav, top_db=30)
        pause_lens = [intervals[i+1][0] - intervals[i][1] for i in range(len(intervals)-1)]
        if pause_lens:
            ind['pause_regularity']  = float(np.std([p/sr*1000 for p in pause_lens]))
            ind['mechanical_pauses'] = bool(np.std(pause_lens) < 20)
        else:
            ind['pause_regularity']  = 0.0
            ind['mechanical_pauses'] = False

        return ind


spectral_analyzer = SpectralAnomalyAnalyzer(cfg)
print(' Explainability Engine ready')

## Risk Scoring & Attack Categorization

In [ ]:
ATTACK_SIGNATURES = {
    'TTS_ATTACK':       {'indicators': ['low_rolloff','robotic_pitch','high_flatness'],    'confidence_boost': 0.10, 'description': 'Text-to-Speech synthesis — unnaturally perfect prosody'},
    'VOICE_CONVERSION': {'indicators': ['low_hnr','unnatural_phase'],                     'confidence_boost': 0.08, 'description': 'Voice conversion — spectral transfer artefacts detected'},
    'REPLAY_ATTACK':    {'indicators': ['unnatural_zcr_variance','high_flatness'],         'confidence_boost': 0.05, 'description': 'Replay attack — recording artefacts and channel noise'},
    'NEURAL_TTS':       {'indicators': ['robotic_pitch','unnatural_phase','low_hnr'],      'confidence_boost': 0.15, 'description': 'Neural TTS (WaveNet/VITS) — highly realistic synthetic'},
    'UNKNOWN_SPOOF':    {'indicators': [],                                                 'confidence_boost': 0.00, 'description': 'Unclassified spoof — anomaly exceeds threshold'},
}


def categorize_attack(indicators: Dict) -> Tuple[str, str]:
    scores = {}
    for attack, info in ATTACK_SIGNATURES.items():
        if not info['indicators']:
            scores[attack] = 0
            continue
        hits = sum(1 for ind in info['indicators'] if indicators.get(ind, False))
        scores[attack] = hits / len(info['indicators'])
    best = max(scores, key=scores.get)
    if scores[best] == 0:
        best = 'UNKNOWN_SPOOF'
    return best, ATTACK_SIGNATURES[best]['description']


def compute_risk_score(fake_prob: float, indicators: Dict, attack_type: str) -> Tuple[float, str]:
    score = fake_prob
    penalty_map = {
        'high_flatness': 0.05, 'low_rolloff': 0.04, 'robotic_pitch': 0.06,
        'unnatural_zcr_variance': 0.04, 'low_hnr': 0.05,
        'unnatural_phase': 0.04, 'mechanical_pauses': 0.03,
    }
    for ind, penalty in penalty_map.items():
        if indicators.get(ind, False):
            score = min(1.0, score + penalty)
    score = min(1.0, score + ATTACK_SIGNATURES.get(attack_type, {}).get('confidence_boost', 0.0))
    level = 'HIGH' if score >= cfg.RISK_HIGH else ('MEDIUM' if score >= cfg.RISK_LOW else 'LOW')
    return round(score, 4), level


def build_forensic_reasons(indicators: Dict, attack_type: str, attack_desc: str) -> List[str]:
    reason_map = {
        'high_flatness':          'Abnormal spectral flatness — characteristic of synthesized speech',
        'low_rolloff':            'Truncated high-frequency content — typical TTS artefact',
        'robotic_pitch':          'Unnaturally stable pitch (F0 std < 8Hz) — indicates synthesis',
        'unnatural_zcr_variance': 'ZCR too uniform — lacks natural speech variability',
        'low_hnr':                'Low harmonic-to-noise ratio — voice cloning spectral distortion',
        'unnatural_phase':        'Phase spectrum anomaly — inconsistent with natural vocal production',
        'mechanical_pauses':      'Pause intervals too regular — unnatural breathing/prosody pattern',
    }
    reasons = [v for k, v in reason_map.items() if indicators.get(k, False)]
    if attack_type != 'UNKNOWN_SPOOF':
        reasons.append(f'Attack signature match: {attack_desc}')
    if not reasons:
        reasons.append('Statistical anomaly — deviates from genuine speech distribution')
    return reasons


print(' Risk Scoring + Attack Categorization ready')

## Speaker Verification Module (ECAPA-TDNN placeholder)

In [ ]:
class ECAPATDNNPlaceholder(nn.Module):
    """Lightweight ECAPA-TDNN-compatible speaker embedding extractor."""
    def __init__(self, in_channels: int = 80, emb_dim: int = 192):
        super().__init__()
        self.frame_enc = nn.Sequential(
            nn.Conv1d(in_channels, 512, 5, padding=2), nn.ReLU(),
            nn.Conv1d(512, 512, 3, padding=1, dilation=1), nn.ReLU(),
            nn.Conv1d(512, 512, 3, padding=2, dilation=2), nn.ReLU(),
        )
        self.attn = nn.Linear(512, 1)
        self.proj = nn.Linear(512 * 2, emb_dim)
        self.norm = nn.LayerNorm(emb_dim)

    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        x    = mel.squeeze(1)
        feat = self.frame_enc(x)
        w    = torch.softmax(self.attn(feat.transpose(1,2)), dim=1)
        mu   = (feat * w.transpose(1,2)).sum(dim=2)
        std  = ((feat**2 * w.transpose(1,2)).sum(dim=2) - mu**2 + 1e-8).sqrt()
        return F.normalize(self.norm(self.proj(torch.cat([mu, std], dim=1))), dim=1)


def verify_speaker_consistency(model, mel_chunks, threshold=0.75) -> Dict:
    model.eval()
    embs = []
    with torch.no_grad():
        for chunk in mel_chunks:
            embs.append(model(chunk.unsqueeze(0).to(DEVICE)))
    if len(embs) < 2:
        return {'consistent': True, 'min_similarity': 1.0, 'mean_similarity': 1.0}
    sims = [float(F.cosine_similarity(embs[0], e).item()) for e in embs[1:]]
    return {
        'consistent':       bool(min(sims) > threshold),
        'min_similarity':   round(min(sims), 4),
        'mean_similarity':  round(float(np.mean(sims)), 4),
        'speaker_shift_risk': 'HIGH' if min(sims) < 0.5 else 'LOW',
    }


speaker_verif = ECAPATDNNPlaceholder(in_channels=cfg.N_MELS).to(DEVICE)
print(f' Speaker Verification | Params: {sum(p.numel() for p in speaker_verif.parameters())/1e6:.2f}M')

## Multi Speaker Detection (pyannote + energy fallback)

In [ ]:
def detect_multi_speaker(audio_path: str, hf_token: Optional[str] = None) -> Dict:
    if hf_token:
        try:
            from pyannote.audio import Pipeline
            pipeline = Pipeline.from_pretrained('pyannote/speaker-diarization-3.1',
                                                 use_auth_token=hf_token)
            pipeline.to(DEVICE)
            diarization = pipeline(audio_path)
            speakers = {s for _, _, s in diarization.itertracks(yield_label=True)}
            n = len(speakers)
            return {'method': 'pyannote', 'n_speakers': n, 'multi_speaker': n > 1,
                    'risk': 'HIGH' if n > 2 else ('MEDIUM' if n > 1 else 'LOW')}
        except Exception as e:
            print(f'pyannote failed ({e}), using fallback')

    try:
        wav, sr    = librosa.load(audio_path, sr=cfg.SAMPLE_RATE)
        frame_len  = int(0.025 * sr)
        hop_len    = int(0.010 * sr)
        energy     = np.array([np.mean(wav[i:i+frame_len]**2)
                                for i in range(0, len(wav)-frame_len, hop_len)])
        from sklearn.cluster import KMeans
        km          = KMeans(n_clusters=2, n_init=3, random_state=42).fit(energy.reshape(-1,1))
        transitions = int(np.sum(np.diff(km.labels_) != 0))
        multi_risk  = transitions > 15
        return {'method': 'energy_fallback', 'transitions': transitions,
                'multi_speaker': multi_risk, 'risk': 'MEDIUM' if multi_risk else 'LOW'}
    except Exception as e:
        return {'method': 'failed', 'error': str(e), 'multi_speaker': False, 'risk': 'LOW'}

print(' Multi-Speaker Detection ready')

## Behavioral Intelligence

In [ ]:
class BehavioralAnalyzer:
    """Detect unnatural prosody, robotic timing, emotion inconsistency."""

    def __init__(self, sr: int = 16000):
        self.sr = sr

    def analyze_prosody(self, wav: np.ndarray) -> Dict:
        results = {}
        rms        = librosa.feature.rms(y=wav, frame_length=512, hop_length=160)[0]
        rms_smooth = np.convolve(rms, np.ones(10)/10, mode='same')
        results['energy_entropy']      = float(-(rms_smooth * np.log(rms_smooth + 1e-8)).sum())
        results['energy_variance']     = float(rms.var())
        results['flat_energy_pattern'] = bool(rms.var() < 0.0005)

        zcr = librosa.feature.zero_crossing_rate(wav, frame_length=512, hop_length=160)[0]
        results['rate_variability']      = float(zcr.std())
        results['robotic_speaking_rate'] = bool(zcr.std() < 0.03)

        centroid = librosa.feature.spectral_centroid(y=wav, sr=self.sr)[0]
        results['centroid_variance']    = float(centroid.var())
        results['emotion_inconsistency'] = bool(centroid.var() < 100000)

        silence_thresh = rms.max() * 0.05
        is_silent      = rms < silence_thresh
        pause_lengths, count, in_pause = [], 0, False
        for s in is_silent:
            if s:
                count += 1; in_pause = True
            elif in_pause:
                pause_lengths.append(count); count = 0; in_pause = False

        if pause_lengths:
            results['mean_pause_frames'] = float(np.mean(pause_lengths))
            results['pause_std']         = float(np.std(pause_lengths))
            results['unnatural_pauses']  = bool(np.std(pause_lengths) < 3 and len(pause_lengths) > 3)
        else:
            results['mean_pause_frames'] = 0.0
            results['pause_std']         = 0.0
            results['unnatural_pauses']  = False

        red_flags = sum([results.get('flat_energy_pattern', False),
                         results.get('robotic_speaking_rate', False),
                         results.get('emotion_inconsistency', False),
                         results.get('unnatural_pauses', False)])
        results['behavioral_risk_score'] = red_flags / 4.0
        results['behavioral_flags']      = red_flags
        return results


behavioral_analyzer = BehavioralAnalyzer(sr=cfg.SAMPLE_RATE)
print(' Behavioral Intelligence ready')

## Autoencoder Anomaly Detection

In [ ]:
class MelAutoencoder(nn.Module):
    """Trained on REAL speech  high reconstruction error → anomalous (fake)."""
    def __init__(self, n_mels: int = 64, latent_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_mels, 128), nn.ReLU(),
            nn.Linear(128, 64),    nn.ReLU(),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 128),        nn.ReLU(),
            nn.Linear(128, n_mels),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        z = self.encoder(x)
        return self.decoder(z), z

    def anomaly_score(self, mel: torch.Tensor) -> float:
        self.eval()
        with torch.no_grad():
            x = mel.squeeze(0).squeeze(0).T.to(DEVICE)
            recon, _ = self(x)
            return round(F.mse_loss(recon, x).item(), 6)


def train_autoencoder(ae, loader, epochs=5):
    optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)
    ae.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0
        for batch in tqdm(loader, desc=f'AE Epoch {epoch}', leave=False):
            if batch is None: continue
            mel, _, labels, _ = batch
            real_mask = labels == 0
            if not real_mask.any(): continue
            real_mel = mel[real_mask].to(DEVICE)
            B, _, F, T_dim = real_mel.shape
            x = real_mel.squeeze(1).permute(0,2,1).contiguous().view(-1, F)
            optimizer.zero_grad()
            recon, _ = ae(x)
            loss = F.mse_loss(recon, x)
            loss.backward(); optimizer.step()
            total_loss += loss.item()
        print(f'  AE Epoch {epoch} | Loss: {total_loss/max(len(loader),1):.6f}')
    return ae


mel_ae = MelAutoencoder(n_mels=cfg.N_MELS, latent_dim=32).to(DEVICE)
if train_loader:
    print('Training Autoencoder on real speech ...')
    mel_ae = train_autoencoder(mel_ae, train_loader, epochs=5)
    torch.save(mel_ae.state_dict(), f'{cfg.DRIVE_PATH}/checkpoints/autoencoder.pt')
print(' Autoencoder ready')

## Real Time Streaming Simulation

In [ ]:
class StreamingInferenceEngine:
    """Chunk-based streaming inference — simulates real-time monitoring."""

    def __init__(self, model, preprocessor, chunk_seconds=2.0, overlap=0.5):
        self.model      = model
        self.prep       = preprocessor
        self.chunk_size = int(chunk_seconds * cfg.SAMPLE_RATE)
        self.hop_size   = int(chunk_seconds * (1 - overlap) * cfg.SAMPLE_RATE)
        self.buffer     = torch.zeros(1, 0)
        self.history    = []

    def process_chunk(self, wav_chunk: torch.Tensor) -> List[Dict]:
        self.buffer = torch.cat([self.buffer, wav_chunk], dim=1)
        results = []
        while self.buffer.shape[1] >= self.chunk_size:
            chunk       = self.buffer[:, :self.chunk_size]
            self.buffer = self.buffer[:, self.hop_size:]
            chunk       = self.prep.normalize(chunk)
            mel         = self.prep.log_mel(chunk).unsqueeze(0).to(DEVICE)
            self.model.eval()
            with torch.no_grad(), autocast():
                logits, _ = self.model(mel)
                prob      = F.softmax(logits, dim=1)[0, 1].item()
            chunk_result = {
                'fake_prob':   round(prob, 4),
                'prediction':  'FAKE' if prob > 0.5 else 'REAL',
                'timestamp_s': len(self.history) * self.hop_size / cfg.SAMPLE_RATE,
            }
            self.history.append(chunk_result)
            results.append(chunk_result)
        return results

    def aggregate(self) -> Dict:
        if not self.history:
            return {'prediction': 'UNKNOWN', 'confidence': 0.0}
        probs      = [c['fake_prob'] for c in self.history]
        final_prob = 0.6 * np.mean(probs) + 0.4 * np.max(probs)
        return {
            'prediction':    'FAKE' if final_prob > 0.5 else 'REAL',
            'confidence':    round(final_prob * 100, 2),
            'n_chunks':      len(self.history),
            'chunk_history': self.history,
        }

    def reset(self):
        self.buffer  = torch.zeros(1, 0)
        self.history = []


def simulate_streaming(audio_path: str, model, preprocessor) -> Dict:
    engine  = StreamingInferenceEngine(model, preprocessor)
    wav, sr = torchaudio.load(audio_path)
    if sr != cfg.SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, cfg.SAMPLE_RATE)
    chunk_sz = int(0.5 * cfg.SAMPLE_RATE)
    for i in range(0, wav.shape[1], chunk_sz):
        engine.process_chunk(wav[:, i:i+chunk_sz])
    return engine.aggregate()

print(' Streaming Engine ready')

## Master Inference Function

In [ ]:
@torch.no_grad()
def analyze_audio(audio_path: str,
                   primary_model=None,
                   secondary_model=None,
                   hf_token: Optional[str] = None) -> Dict:
    """Full enterprise voice forensics pipeline."""
    primary_model   = primary_model   or aasist_model
    secondary_model = secondary_model or cnn_model

    feats = preprocessor.process(audio_path)
    if feats is None:
        return {'error': 'Could not load audio file'}

    mel    = feats['log_mel'].unsqueeze(0).to(DEVICE)
    wav_np = feats['wav'].squeeze().numpy()

    primary_model.eval(); secondary_model.eval()
    with autocast():
        logits1, emb1 = primary_model(mel)
        logits2, emb2 = secondary_model(mel)

    prob1      = F.softmax(logits1, dim=1)[0, 1].item()
    prob2      = F.softmax(logits2, dim=1)[0, 1].item()
    fake_prob  = round(0.6 * prob1 + 0.4 * prob2, 4)
    prediction = 'FAKE' if fake_prob > 0.5 else 'REAL'
    confidence = round(fake_prob * 100 if prediction == 'FAKE' else (1 - fake_prob) * 100, 2)

    spec_indicators         = spectral_analyzer.analyze(wav_np, sr=cfg.SAMPLE_RATE)
    behavioral              = behavioral_analyzer.analyze_prosody(wav_np)
    attack_cat, attack_desc = categorize_attack(spec_indicators)
    risk_score, risk_level  = compute_risk_score(fake_prob, spec_indicators, attack_cat)

    beh_risk   = behavioral.get('behavioral_risk_score', 0.0)
    risk_score = min(1.0, risk_score + 0.1 * beh_risk)
    risk_level = 'HIGH' if risk_score >= cfg.RISK_HIGH else ('MEDIUM' if risk_score >= cfg.RISK_LOW else 'LOW')

    reasons = build_forensic_reasons(spec_indicators, attack_cat, attack_desc)
    if behavioral.get('flat_energy_pattern'):   reasons.append('Unnaturally flat energy contour')
    if behavioral.get('robotic_speaking_rate'): reasons.append('Robotic speaking rate regularity')
    if behavioral.get('emotion_inconsistency'): reasons.append('Emotional inconsistency in spectral timbre')
    if behavioral.get('unnatural_pauses'):      reasons.append('Mechanically timed pause intervals')

    mel_chunks    = [feats['log_mel'][:,:,i:i+50] for i in range(0, feats['log_mel'].shape[2]-50, 50)][:5]
    speaker_result = verify_speaker_consistency(speaker_verif, mel_chunks)
    multi_spk      = detect_multi_speaker(audio_path, hf_token=hf_token)
    ae_score       = mel_ae.anomaly_score(mel)
    stream_result  = simulate_streaming(audio_path, primary_model, preprocessor)

    return {
        'prediction':          prediction,
        'confidence':          confidence,
        'risk_level':          risk_level,
        'risk_score':          round(risk_score * 100, 2),
        'attack_category':     attack_cat,
        'attack_description':  attack_desc,
        'reasons':             reasons,
        'spectral_indicators': {k: v for k, v in spec_indicators.items() if isinstance(v, bool)},
        'behavioral_flags':    behavioral.get('behavioral_flags', 0),
        'speaker_consistency': speaker_result,
        'multi_speaker':       multi_spk,
        'anomaly_score':       ae_score,
        'streaming':           stream_result,
        'model_scores': {
            'aasist_fake_prob': round(prob1, 4),
            'cnn_fake_prob':    round(prob2, 4),
            'ensemble_prob':    fake_prob,
        },
    }

print(' Master inference function ready')

## Visualization Dashboard

In [ ]:
def visualize_forensics(audio_path: str, report: Dict, save: bool = True):
    """Generate full forensics visualization dashboard."""
    wav, sr = librosa.load(audio_path, sr=cfg.SAMPLE_RATE)
    feats   = preprocessor.process(audio_path)

    fig = plt.figure(figsize=(20, 18))
    fig.patch.set_facecolor('#0d1117')
    color_real = '#22c55e'
    color_fake = '#ef4444'
    main_color = color_fake if report['prediction'] == 'FAKE' else color_real
    plt.rcParams['text.color'] = 'white'
    gs = fig.add_gridspec(4, 3, hspace=0.45, wspace=0.35)

    ax_title = fig.add_subplot(gs[0, :])
    ax_title.set_facecolor('#161b22')
    risk_emoji = {'LOW', 'MEDIUM', 'HIGH'}
    ax_title.text(0.5, 0.65,
        f" VOICE FORENSICS REPORT  |  {report['prediction']}  |  {report['confidence']:.1f}% confidence",
        ha='center', va='center', fontsize=15, fontweight='bold',
        color=main_color, transform=ax_title.transAxes)
    ax_title.text(0.5, 0.25,
        f"Risk: {risk_emoji.get(report['risk_level'],'?')} {report['risk_level']} ({report['risk_score']:.1f}%)  |  "
        f"Attack: {report['attack_category']}  |  Anomaly: {report['anomaly_score']:.5f}",
        ha='center', va='center', fontsize=11, color='#94a3b8', transform=ax_title.transAxes)
    ax_title.axis('off')

    ax_wav = fig.add_subplot(gs[1, :])
    ax_wav.set_facecolor('#161b22')
    t = np.linspace(0, len(wav)/sr, len(wav))
    ax_wav.plot(t, wav, color=main_color, linewidth=0.4, alpha=0.8)
    ax_wav.fill_between(t, wav, alpha=0.15, color=main_color)
    ax_wav.set_title('Waveform', color='white', fontsize=10)
    ax_wav.set_xlabel('Time (s)', color='#64748b')
    ax_wav.tick_params(colors='#64748b')
    for sp in ax_wav.spines.values(): sp.set_edgecolor('#30363d')

    ax_mel = fig.add_subplot(gs[2, 0])
    ax_mel.set_facecolor('#161b22')
    if feats:
        im = ax_mel.imshow(feats['log_mel'].squeeze().numpy(), aspect='auto', origin='lower', cmap='inferno')
        plt.colorbar(im, ax=ax_mel, fraction=0.03)
    ax_mel.set_title('Log-Mel Spectrogram', color='white', fontsize=10)
    ax_mel.tick_params(colors='#64748b')

    ax_cqcc = fig.add_subplot(gs[2, 1])
    ax_cqcc.set_facecolor('#161b22')
    if feats:
        im2 = ax_cqcc.imshow(feats['cqcc'].squeeze().numpy(), aspect='auto', origin='lower', cmap='viridis')
        plt.colorbar(im2, ax=ax_cqcc, fraction=0.03)
    ax_cqcc.set_title('CQCC Features', color='white', fontsize=10)
    ax_cqcc.tick_params(colors='#64748b')

    ax_risk = fig.add_subplot(gs[2, 2])
    ax_risk.set_facecolor('#161b22')
    ax_risk.barh(['Risk Score'], [report['risk_score']/100], color=main_color, height=0.4)
    ax_risk.axvline(cfg.RISK_LOW,  color='#eab308', linestyle='--', lw=1.5, label='Medium')
    ax_risk.axvline(cfg.RISK_HIGH, color='#ef4444', linestyle='--', lw=1.5, label='High')
    ax_risk.set_xlim(0, 1)
    ax_risk.set_title('Enterprise Risk Score', color='white', fontsize=10)
    ax_risk.tick_params(colors='#64748b')
    ax_risk.legend(fontsize=8, facecolor='#161b22', labelcolor='white')
    for sp in ax_risk.spines.values(): sp.set_edgecolor('#30363d')

    ax_stream = fig.add_subplot(gs[3, :2])
    ax_stream.set_facecolor('#161b22')
    hist = report.get('streaming', {}).get('chunk_history', [])
    if hist:
        ts    = [c['timestamp_s'] for c in hist]
        probs = [c['fake_prob'] for c in hist]
        ax_stream.plot(ts, probs, color=main_color, lw=2, marker='o', markersize=3)
        ax_stream.fill_between(ts, probs, alpha=0.2, color=main_color)
        ax_stream.axhline(0.5, color='#64748b', linestyle='--', lw=1)
    ax_stream.set_title('Streaming Fake Probability Timeline', color='white', fontsize=10)
    ax_stream.set_xlabel('Time (s)', color='#64748b')
    ax_stream.set_ylabel('Fake Prob', color='#64748b')
    ax_stream.tick_params(colors='#64748b')
    ax_stream.set_ylim(0, 1)
    for sp in ax_stream.spines.values(): sp.set_edgecolor('#30363d')

    ax_reasons = fig.add_subplot(gs[3, 2])
    ax_reasons.set_facecolor('#161b22')
    ax_reasons.axis('off')
    ax_reasons.set_title('Forensic Indicators', color='white', fontsize=10)
    reasons_text = '\n'.join([f'• {r[:55]}' for r in report['reasons'][:6]])
    ax_reasons.text(0.05, 0.9, reasons_text, transform=ax_reasons.transAxes,
                     va='top', ha='left', fontsize=7.5, color='#94a3b8', linespacing=2.0)

    if save:
        out = f"{cfg.DRIVE_PATH}/forensics/report_{Path(audio_path).stem}.png"
        plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
        print(f'  Saved → {out}')
    plt.show()

print(' Visualization dashboard ready')

## GradCAM Heatmap Visualization

In [ ]:
def visualize_gradcam(audio_path: str, model: CNNBaseline):
    """Show GradCAM heatmap overlaid on Mel spectrogram."""
    feats = preprocessor.process(audio_path)
    if feats is None: return
    mel     = feats['log_mel'].unsqueeze(0)
    gradcam = GradCAM(model, target_layer_idx=-1)
    cam     = gradcam(mel, class_idx=1)
    mel_img = mel.squeeze().detach().numpy()

    from scipy.ndimage import zoom
    cam_resized = zoom(cam, (mel_img.shape[0]/(cam.shape[0]+1e-8),
                              mel_img.shape[1]/(cam.shape[1]+1e-8)), order=1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.patch.set_facecolor('#0d1117')
    axes[0].imshow(mel_img, aspect='auto', origin='lower', cmap='inferno')
    axes[0].set_title('Log-Mel Spectrogram', color='white')
    axes[0].tick_params(colors='gray')
    axes[1].imshow(mel_img, aspect='auto', origin='lower', cmap='inferno')
    axes[1].imshow(cam_resized, aspect='auto', origin='lower', cmap='jet', alpha=0.5)
    axes[1].set_title('GradCAM — Suspicious Regions', color='white')
    axes[1].tick_params(colors='gray')
    for ax in axes:
        ax.set_facecolor('#161b22')
        for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
    plt.tight_layout()
    out = f"{cfg.DRIVE_PATH}/forensics/gradcam_{Path(audio_path).stem}.png"
    plt.savefig(out, dpi=150, facecolor=fig.get_facecolor())
    plt.show()
    print(f'Saved GradCAM → {out}')

print(' GradCAM ready')

## Save & Load Models

In [ ]:
def save_models():
    torch.save({
        'cnn_state':    cnn_model.state_dict(),
        'aasist_state': aasist_model.state_dict(),
        'ae_state':     mel_ae.state_dict(),
        'config':       cfg.__dict__,
    }, f'{cfg.DRIVE_PATH}/checkpoints/full_platform.pt')
    print(f' All models saved → {cfg.DRIVE_PATH}/checkpoints/full_platform.pt')


def load_models(path: str = None):
    path = path or f'{cfg.DRIVE_PATH}/checkpoints/full_platform.pt'
    if not os.path.exists(path):
        print(f'Checkpoint not found: {path}'); return
    ckpt = torch.load(path, map_location=DEVICE, weights_only=True)
    cnn_model.load_state_dict(ckpt['cnn_state'])
    aasist_model.load_state_dict(ckpt['aasist_state'])
    mel_ae.load_state_dict(ckpt['ae_state'])
    print(f' Models loaded from {path}')


save_models()
# load_models()  # ← Uncomment to restore from Drive

## Test on Eval Sample

In [ ]:
def test_custom_audio(audio_path: str, hf_token: Optional[str] = None):
    """Run full pipeline on any audio file and print forensic report."""
    print(f'\n Analyzing: {audio_path}')
    print('─' * 60)
    report = analyze_audio(audio_path, hf_token=hf_token)
    if 'error' in report:
        print(f' Error: {report["error"]}'); return

    verdict_icon = ' FAKE' if report['prediction'] == 'FAKE' else ' REAL'
    risk_icon    = {'LOW', 'MEDIUM', 'HIGH'}
    print(f'  Verdict:          {verdict_icon}  ({report["confidence"]}% confidence)')
    print(f'  Risk Level:       {risk_icon.get(report["risk_level"],"?")} {report["risk_level"]} ({report["risk_score"]}%)')
    print(f'  Attack Category:  {report["attack_category"]}')
    print(f'  Anomaly Score:    {report["anomaly_score"]}')
    print(f'  Behavioral Flags: {report["behavioral_flags"]}/4')
    print(f'  Multi-Speaker:    {report["multi_speaker"].get("multi_speaker","?")} ({report["multi_speaker"].get("risk","?")} risk)')
    print('\n   Forensic Reasons:')
    for r in report['reasons']:
        print(f'    • {r}')
    print('\n   Model Ensemble:')
    for k, v in report['model_scores'].items():
        print(f'    {k}: {v}')

    visualize_forensics(audio_path, report)
    visualize_gradcam(audio_path, cnn_model)

    json_path = f"{cfg.DRIVE_PATH}/forensics/report_{Path(audio_path).stem}.json"
    with open(json_path, 'w') as f:
        json.dump({k: v for k, v in report.items() if k != 'spectral_indicators'}, f, indent=2)
    print(f'\n   JSON report saved → {json_path}')
    return report


# Test on first eval file
if eval_records:
    sample = eval_records[0]
    print(f'Testing on: {sample["audio_path"]}')
    print(f'True label: {sample["label_str"]}  |  Language: {sample["language"]}')
    report = test_custom_audio(sample['audio_path'])
else:
    print('No eval records — upload your own audio below')

## Test on Your Own Audio File

In [ ]:
from google.colab import files as colab_files

print('Upload a .wav or .flac file to analyze:')
uploaded_audio = colab_files.upload()

for filename in uploaded_audio:
    save_path = f'/content/{filename}'
    with open(save_path, 'wb') as f:
        f.write(uploaded_audio[filename])
    test_custom_audio(save_path)

## Training History Visualization

In [ ]:
def plot_training_history(history: Dict, model_name: str = 'Model'):
    if not history or not history.get('train_loss'):
        print('No training history to plot'); return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0d1117')
    for ax in axes:
        ax.set_facecolor('#161b22')
        for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
        ax.tick_params(colors='gray')

    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], '#3b82f6', lw=2, label='Train')
    axes[0].plot(epochs, history['val_loss'],   '#ef4444', lw=2, label='Val')
    axes[0].set_title(f'{model_name} Loss', color='white')
    axes[0].set_xlabel('Epoch', color='gray'); axes[0].set_ylabel('Loss', color='gray')
    axes[0].legend(facecolor='#161b22', labelcolor='white')

    axes[1].plot(epochs, history['train_acc'], '#3b82f6', lw=2, label='Train')
    axes[1].plot(epochs, history['val_acc'],   '#22c55e', lw=2, label='Val')
    axes[1].set_title(f'{model_name} Accuracy', color='white')
    axes[1].set_xlabel('Epoch', color='gray'); axes[1].set_ylabel('Accuracy', color='gray')
    axes[1].legend(facecolor='#161b22', labelcolor='white')

    plt.tight_layout()
    plt.savefig(f'{cfg.DRIVE_PATH}/forensics/{model_name}_history.png', dpi=150,
                facecolor=fig.get_facecolor())
    plt.show()


try:
    plot_training_history(cnn_history,    'CNN_Baseline')
    plot_training_history(aasist_history, 'AASIST')
except NameError:
    print('Training history not available — train models first')

## Platform Summary

In [ ]:
print('\n' + '='*60)
print('  AI Voice Trust Platform — IndicVoices Edition')
print('='*60)
print(f'  Device         : {DEVICE}')
print(f'  Real samples   : {len(real_records)} (IndicVoices — genuine human speech)')
print(f'  Fake samples   : {len(fake_records)} (augmentation-generated spoofs)')
print(f'  Languages      : {len(INDIC_LANGS)} ({", ".join(INDIC_LANGS[:5])} ...)')
print(f'  CNN Baseline   : {sum(p.numel() for p in cnn_model.parameters())/1e6:.2f}M params')
print(f'  AASIST         : {sum(p.numel() for p in aasist_model.parameters())/1e6:.2f}M params')
print(f'  Speaker Verif  : {sum(p.numel() for p in speaker_verif.parameters())/1e6:.2f}M params')
print(f'  Autoencoder    : {sum(p.numel() for p in mel_ae.parameters())/1e6:.2f}M params')
print(f'  Outputs saved  : {cfg.DRIVE_PATH}')
print('='*60)